## 02 - Silver Transform: orders

Limpeza da tabela `orders` (camada Bronze → Silver)

In [0]:
%python
from pyspark.sql.functions import col, to_timestamp

orders_bronze = spark.table("olist_project.bronze.orders")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

required_columns = ["order_id", "customer_id"]

orders_silver = orders_bronze.dropDuplicates(["order_id"])

for date_col in date_columns:
    orders_silver = orders_silver.withColumn(date_col, to_timestamp(col(date_col)))

for required_col in required_columns:
    orders_silver = orders_silver.filter(col(required_col).isNotNull())

(
    orders_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.orders")
)

print(f"orders_silver: {orders_silver.count()} linhas")